# LPKIA Scholarship Rule Engine – Decision Tree

Notebook ini membangun model **Decision Tree** (criterion `entropy`, `max_depth=4`, `min_samples_leaf=5`) untuk memprediksi kelayakan beasiswa, lalu mengubahnya menjadi aturan **IF-THEN** (production rules) dengan format JSON yang sama seperti pada `lpkia_rule_engine.html`.

Fitur: `IPK`, `Penghasilan_Ortu` (juta Rp/bulan), `Jumlah_Tanggungan` (1–5), `Keaktifan_Organisasi` (0 = tidak aktif, 1 = aktif, 2 = pengurus inti).
Target: `Layak` (1 = LAYAK, 0 = TIDAK LAYAK).

> Jika Anda punya data asli, simpan sebagai `data_beasiswa.csv` di folder yang sama. Jika tidak ada, notebook akan membuat data sintetis 400 baris sebagai contoh.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
FITUR = ['IPK', 'Penghasilan_Ortu', 'Jumlah_Tanggungan', 'Keaktifan_Organisasi']
TARGET = 'Layak'

## 1. Memuat data

In [ ]:
def buat_data_sintetis(n=400, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    ipk = np.round(rng.uniform(2.0, 4.0, n), 2)
    income = np.round(rng.gamma(shape=3.0, scale=2.5, size=n), 1)
    tanggungan = rng.integers(1, 6, n)
    org = rng.choice([0, 1, 2], size=n, p=[0.35, 0.45, 0.20])

    # skor kelayakan: IPK tinggi, penghasilan rendah, tanggungan banyak, aktif organisasi
    skor = (
        2.2 * (ipk - 3.0)
        - 0.35 * (income - 6.0)
        + 0.25 * (tanggungan - 3)
        + 0.35 * org
        + rng.normal(0, 0.6, n)
    )
    layak = (skor > 0).astype(int)
    return pd.DataFrame({
        'IPK': ipk,
        'Penghasilan_Ortu': income,
        'Jumlah_Tanggungan': tanggungan,
        'Keaktifan_Organisasi': org,
        TARGET: layak,
    })

if os.path.exists('data_beasiswa.csv'):
    df = pd.read_csv('data_beasiswa.csv', sep=None, engine='python')
    print('Data dimuat dari data_beasiswa.csv')
else:
    df = buat_data_sintetis()
    print('data_beasiswa.csv tidak ditemukan -> memakai data sintetis')

print(df.shape)
df.head()

In [ ]:
print(df.describe().round(2))
print('\nDistribusi target:')
print(df[TARGET].value_counts().rename({1: 'LAYAK', 0: 'TIDAK LAYAK'}))
print('\nMissing value per kolom:')
print(df.isna().sum())

## 2. Entropy awal (root) dan pembagian data

In [ ]:
def entropy(y):
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

print(f'Entropy root: {entropy(df[TARGET].values):.4f}')

X = df[FITUR]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 3. Melatih Decision Tree

In [ ]:
model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=4,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

print('Kedalaman pohon :', model.get_depth())
print('Jumlah daun     :', model.get_n_leaves())

imp = pd.Series(model.feature_importances_, index=FITUR).sort_values(ascending=False)
print('\nFeature importance:')
print(imp.round(4))

## 4. Evaluasi

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Akurasi data uji: {acc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=['TIDAK LAYAK', 'LAYAK']))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['TIDAK LAYAK', 'LAYAK']).plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 5. Visualisasi pohon keputusan

In [ ]:
plt.figure(figsize=(26, 10))
plot_tree(
    model,
    feature_names=FITUR,
    class_names=['TIDAK LAYAK', 'LAYAK'],
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title('Decision Tree - Kelayakan Beasiswa LPKIA')
plt.show()

print(export_text(model, feature_names=FITUR))

## 6. Ekstraksi aturan IF-THEN (production rules)

Setiap jalur dari root ke daun menjadi satu aturan. Probabilitas adalah proporsi kelas yang diprediksi pada daun tersebut.

In [ ]:
def ekstrak_aturan(clf, nama_fitur):
    t = clf.tree_
    hasil = []

    def telusuri(node, kondisi):
        if t.feature[node] != -2:  # node internal
            f = nama_fitur[t.feature[node]]
            th = t.threshold[node]
            telusuri(t.children_left[node], kondisi + [f'{f} <= {th:.2f}'])
            telusuri(t.children_right[node], kondisi + [f'{f} > {th:.2f}'])
        else:  # daun
            v = t.value[node][0]
            v = v / v.sum()
            kelas = int(clf.classes_[v.argmax()])
            hasil.append({
                'if': kondisi,
                'then': 'LAYAK' if kelas == 1 else 'TIDAK LAYAK',
                'probability': round(float(v.max()), 3),
                'samples': int(t.n_node_samples[node]),
            })

    telusuri(0, [])
    return hasil

aturan = ekstrak_aturan(model, FITUR)

for i, r in enumerate(aturan, 1):
    print(f"R{i:<2} IF {' AND '.join(r['if'])}")
    print(f"     THEN {r['then']}  (P={r['probability']}, n={r['samples']})\n")

## 7. Ekspor aturan ke JSON (untuk rule engine HTML)

In [ ]:
payload = {
    'criterion': model.criterion,
    'max_depth': model.max_depth,
    'min_samples_leaf': model.min_samples_leaf,
    'root_entropy': round(entropy(df[TARGET].values), 4),
    'accuracy': round(float(acc), 4),
    'n_rules': len(aturan),
    'rules': aturan,
}

with open('rules.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print('Tersimpan: rules.json')
print('Salin isinya ke konstanta `const D=` pada lpkia_rule_engine.html')

## 8. Fungsi prediksi berbasis aturan (setara logika JavaScript)

In [ ]:
import re

POLA = re.compile(r'^(\w+) (<=|>) ([\d.]+)$')

def prediksi_rule(x, rules=aturan):
    """x: dict berisi 4 fitur. Mengembalikan (label, P_layak, nomor_aturan)."""
    for i, r in enumerate(rules, 1):
        cocok = True
        for c in r['if']:
            f, op, v = POLA.match(c).groups()
            v = float(v)
            if not (x[f] <= v if op == '<=' else x[f] > v):
                cocok = False
                break
        if cocok:
            p = r['probability'] if r['then'] == 'LAYAK' else 1 - r['probability']
            return r['then'], round(p, 3), i
    return None

def validasi(x):
    if not 2 <= x['IPK'] <= 4: return 'IPK harus 2.00 - 4.00'
    if x['Penghasilan_Ortu'] < 0: return 'Penghasilan tidak boleh negatif'
    if x['Jumlah_Tanggungan'] not in range(1, 6): return 'Tanggungan harus 1 - 5'
    if x['Keaktifan_Organisasi'] not in (0, 1, 2): return 'Keaktifan organisasi harus 0, 1, atau 2'
    return None

# Contoh satu pendaftar
pendaftar = {'IPK': 3.40, 'Penghasilan_Ortu': 4.5, 'Jumlah_Tanggungan': 3, 'Keaktifan_Organisasi': 1}
err = validasi(pendaftar)
if err:
    print('Error:', err)
else:
    label, p, no = prediksi_rule(pendaftar)
    print(f'{label} | P(Layak)={p:.1%} | aturan R{no}')

# Verifikasi: hasil rule = hasil model sklearn pada seluruh data uji
cocok = sum(
    (prediksi_rule(row.to_dict())[0] == 'LAYAK') == bool(pred)
    for (_, row), pred in zip(X_test.iterrows(), y_pred)
)
print(f'Kecocokan rule vs model: {cocok}/{len(X_test)}')

## 9. Uji banyak pendaftar dari CSV

Kolom wajib: `IPK, Penghasilan_Ortu, Jumlah_Tanggungan, Keaktifan_Organisasi` (kolom `Nama` opsional).

In [ ]:
contoh = pd.DataFrame({
    'Nama': ['Ayu', 'Bagas', 'Citra', 'Dimas'],
    'IPK': [3.6, 2.4, 3.1, 3.8],
    'Penghasilan_Ortu': [3.0, 12.0, 6.0, 10.5],
    'Jumlah_Tanggungan': [4, 1, 3, 2],
    'Keaktifan_Organisasi': [2, 0, 1, 2],
})
contoh.to_csv('contoh_pendaftar.csv', index=False)

def uji_csv(path, keluaran='hasil_prediksi.csv'):
    d = pd.read_csv(path, sep=None, engine='python')
    d.columns = [c.strip() for c in d.columns]
    kurang = [c for c in FITUR if c not in d.columns]
    if kurang:
        raise ValueError(f'Kolom tidak ditemukan: {kurang}')

    baris = []
    for i, row in d.iterrows():
        x = {f: float(row[f]) for f in FITUR}
        err = validasi(x)
        if err:
            print(f'Baris {i+1} dilewati: {err}')
            continue
        label, p, no = prediksi_rule(x)
        baris.append({**row.to_dict(), 'Prediksi': label, 'P_Layak': p, 'Aturan': f'R{no}'})

    hasil = pd.DataFrame(baris)
    hasil.to_csv(keluaran, index=False)
    print(f'Diproses: {len(hasil)} | Layak: {(hasil.Prediksi == "LAYAK").sum()} | '
          f'Tidak layak: {(hasil.Prediksi == "TIDAK LAYAK").sum()}')
    return hasil

uji_csv('contoh_pendaftar.csv')

## 10. Menyimpan model

In [ ]:
import joblib
joblib.dump(model, 'model_decision_tree_beasiswa.joblib')
print('Tersimpan: model_decision_tree_beasiswa.joblib')